# Eth Nl2Sql Fine Tune

Generated from local source. Edit src/ and scripts/, then run `eth-nl2sql kaggle build`; deploy this exact notebook.

- Kernel: `aohkvd/eth-nl2sql-fine-tune`
- Git commit: `5b8fbd8896fa0278aced75a36f3628e0b1d10317`
- Includes uncommitted source: `False`
- Job / config / seed: `b3-seed-42` / `b3` / `42`
- Entry point: `scripts/t11b.py`
- Input dataset: `aohkvd/eth-nl2sql-inputs`
- Source SHA256: `8f61d1874a0d7ffe2fe6196860d41345d93e391585ce9d4aaea96cfec4658270`
- Data SHA256: `5e1ce1f6c5aa331ac366e7c53d61311d6f767b4b933d4739af0f03dd2ea00244`
- Input ZIP SHA256: `d5a12df43f944154e164b1326349a90990b42c52f3b168f4bbc1e03a30e1ee99`

Only questions and IDs from dev_eval are sent to Kaggle; evaluation stays local.

In [ ]:
INPUT_FILE = 'inputs-d5a12df43f944154e164b1326349a90990b42c52f3b168f4bbc1e03a30e1ee99.bundle'
INPUT_SHA256 = 'd5a12df43f944154e164b1326349a90990b42c52f3b168f4bbc1e03a30e1ee99'
PROVENANCE = {'code_commit': '5b8fbd8896fa0278aced75a36f3628e0b1d10317',
 'config': 'b3',
 'data_sha256': '5e1ce1f6c5aa331ac366e7c53d61311d6f767b4b933d4739af0f03dd2ea00244',
 'input_sha256': 'd5a12df43f944154e164b1326349a90990b42c52f3b168f4bbc1e03a30e1ee99',
 'job': 'b3-seed-42',
 'script': 'scripts/t11b.py',
 'seed': 42,
 'source_sha256': '8f61d1874a0d7ffe2fe6196860d41345d93e391585ce9d4aaea96cfec4658270',
 'working_tree_changes': False}


## Verify and unpack attached inputs

In [ ]:
import hashlib, json, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

matches = list(Path('/kaggle/input').rglob(INPUT_FILE))
if len(matches) != 1:
    raise ValueError('Attach the matching workflow inputs dataset: ' + INPUT_FILE)
archive = matches[0]
h = hashlib.sha256()
with archive.open('rb') as handle:
    while chunk := handle.read(8 * 1024 * 1024):
        h.update(chunk)
if h.hexdigest() != INPUT_SHA256:
    raise ValueError('Input archive checksum mismatch; rebuild and publish matching inputs')
root = Path('/kaggle/working/eth-nl2sql')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        if not (root / member.filename).resolve().is_relative_to(root):
            raise ValueError('Invalid bundle path')
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Symlink in input bundle')
    bundle.extractall(root)
print('Verified inputs:', INPUT_SHA256)
print('Code commit:', PROVENANCE.get('code_commit'))
print('Source fingerprint:', PROVENANCE.get('source_sha256'))


## Install the locked environment

In [ ]:
scratch = Path(tempfile.mkdtemp(prefix='workflow-runtime-', dir='/tmp'))
log_root = Path('/kaggle/working/t11b-bootstrap')
log_root.mkdir()

def run(stage, argv, env=None):
    print('Starting:', stage, flush=True)
    with (log_root / (stage + '.log')).open('w') as log:
        process = subprocess.Popen(argv, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        result = subprocess.CompletedProcess(argv, process.wait())
    if result.returncode:
        print((log_root / (stage + '.log')).read_text()[-6000:], flush=True)
        raise RuntimeError(stage + ' failed')
    print('Completed:', stage, flush=True)

run('bootstrap-uv', [sys.executable, '-m', 'pip', '--isolated', 'install',
    '--no-deps', '--target', str(scratch / 'bootstrap'), 'uv==0.11.14'])
uv = str(scratch / 'bootstrap/bin/uv')
runtime = scratch / 'runtime'
run('python311', [uv, '--no-config', 'venv', '--python', '3.11.15',
    '--managed-python', str(runtime)])
python = str(runtime / 'bin/python')
run('install-locked-environment', [uv, '--no-config', '--no-cache', 'pip', 'sync',
    '--python', python, '--require-hashes', '--index-url', 'https://pypi.org/simple',
    str(root / 'configs/t11b-gpu-requirements.txt')])


## Run the development experiment

In [ ]:
matches = list(Path('/kaggle/input').rglob('models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json'))
if len(matches) != 1:
    raise ValueError('Expected one attached pinned model cache: ' + str(matches))
home = matches[0].parents[4]
env = dict(os.environ, PYTHONPATH=str(root / 'src'), OMP_NUM_THREADS='2',
    HF_HOME=str(home), HF_HUB_CACHE=str(home / 'hub'), HF_HUB_OFFLINE='1',
    TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false',
    HF_HUB_DISABLE_IMPLICIT_TOKEN='1', WANDB_DISABLED='true')
run('gpu-pipeline', [python, str(root / 'scripts/t11b.py'), 'gpu',
    '--job', 'b3-seed-42', '--hf-home', str(home)], env)
print('Copy t11b-b3-seed-42-artifacts.zip back intact for local development EX.', flush=True)
